In [0]:
table_name=[]

In [0]:
for i in dbutils.fs.ls("abfss://bronze@projectadlsgen2demo0506.dfs.core.windows.net/Production/"):
    table_name.append(i.name.split("/")[0])

In [0]:
table_name

In [0]:
from pyspark.sql.functions import date_format, from_utc_timestamp
from pyspark.sql.types import TimestampType

for i in table_name:

    input_path = f"abfss://bronze@projectadlsgen2demo0506.dfs.core.windows.net/Production/{i}/{i}.parquet"

    df = spark.read.format("parquet").load(input_path)

    for col in df.columns:
        if "Date" in col or "date" in col:
            df = df.withColumn(
                col,
                date_format(
                    from_utc_timestamp(df[col].cast(TimestampType()), "UTC"),
                    "yyyy-MM-dd HH:mm:ss"
                )
            )

    output_path = f"abfss://silver@projectadlsgen2demo0506.dfs.core.windows.net/{i}/"

    df.write.mode("overwrite").format("delta").save(output_path)

In [0]:
df.printSchema()

In [0]:
df.display()